In [1]:
import sys  # handle with python interpreter
from pathlib import Path # Handle file paths
import pandas as pd

ROOT = Path.cwd().resolve().parent # Find the root directory, resolve() to get absolute path, parent to go up one level
# Instead of /DL4AI-240166-project-1/notebooks, we want /DL4AI-240166-project-1
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Now Python now how to get the src/data/preprocess.py file

In [2]:
from src.data.preprocess import add_all_features

In [ ]:

# Đường dẫn folder chứa các file csv của Vietnam
DATA_DIR = ROOT / 'notebooks' / 'data' / 'vietnam' 

# 2. LOAD CÁC FILE THÀNH PHẦN
sentiment_df = pd.read_csv(DATA_DIR / 'final_sentiment_features_v1.csv')
fundamentals_df = pd.read_csv(DATA_DIR / 'finance_indicators.csv')

# 3. CHUẨN HÓA FUNDAMENTALS (Xử lý tên cột và Ticker)
# Xóa khoảng trắng thừa và chuẩn hóa tên cột năm
fundamentals_df.columns = [c.strip() for c in fundamentals_df.columns]
if 'yearReport' in fundamentals_df.columns:
    fundamentals_df = fundamentals_df.rename(columns={'yearReport': 'year'})

cols_to_drop = ['sharesOutstanding', 'netProfit', 'npl']
for col in cols_to_drop:
    if col in fundamentals_df.columns:
        fundamentals_df = fundamentals_df.drop(columns=[col])

fundamentals_df['ticker'] = fundamentals_df['ticker'].str.upper()

#  QUY TRÌNH XỬ LÝ LẶP TỪNG MÃ (Sử dụng file lẻ trong folder csv)
frames = []
vn_tickers = ['FPT', 'VCB', 'VIC', 'VHM', 'HPG', 'MSN', 'MWG', 'TCB', 'VND', 'VNM']

for ticker in vn_tickers:
    file_path = DATA_DIR / 'csv' / f'{ticker}_ohlcv.csv'
    if not file_path.exists():
        continue
    
    print(f"Merging features for: {ticker}")
    
    # A. Load và xử lý OHLCV
    g = pd.read_csv(file_path)
    g = g.rename(columns={'tradingDate': 'date'})
    g["date"] = pd.to_datetime(g["date"])
    g = g.sort_values("date").set_index("date")
    
    # B. Tính Technical Features (RSI, MACD, Log Return...)
    g = add_all_features(g) 
    g = g.reset_index()
    g["ticker"] = ticker
    
    # C. Merge Sentiment (Key: Date)
    t_sent = sentiment_df[sentiment_df["ticker"] == ticker].copy()
    t_sent["date"] = pd.to_datetime(t_sent["date"])
    g = pd.merge(g, t_sent, on=["date", "ticker"], how="left")
    
    # D. Merge Fundamentals (Key: Year)
    g['year'] = g['date'].dt.year
    t_fund = fundamentals_df[fundamentals_df["ticker"] == ticker].copy()

    # THÊM DÒNG NÀY: Loại bỏ trùng lặp trong dữ liệu tài chính của ticker hiện tại
    t_fund = t_fund.drop_duplicates(subset=['year']) 
    
    g = pd.merge(g, t_fund, on=["year", "ticker"], how="left")
    
    # Merge qua Year để lấy PE, PB, ROE, ROA, Asset, BV, Beta, leverage...
    g = pd.merge(g, t_fund, on=["year", "ticker"], how="left")
    
    # E. Làm sạch dữ liệu sau merge
    # Forward fill dữ liệu tài chính cho các ngày trong năm
    g = g.ffill()
    
    # Điền 0 cho News Sentiment nếu ngày đó không có tin
    sent_cols = ['sent_mean', 'sent_max', 'sent_min', 'news_count']
    for col in sent_cols:
        if col in g.columns:
            g[col] = g[col].fillna(0)
    
    frames.append(g)

# 5. GỘP TẤT CẢ VÀ KIỂM TRA LỖI LẦN CUỐI
df_vn_master = pd.concat(frames, ignore_index=True)

# Khử NaN ở đầu chuỗi (do các chỉ số kỹ thuật)
df_vn_master = df_vn_master.bfill()

# Xóa cột year tạm thời
if 'year' in df_vn_master.columns:
    df_vn_master = df_vn_master.drop(columns=['year'])

# 6. LƯU FILE MASTER
out_path = DATA_DIR / "vn_stocks_master_features.csv"
df_vn_master.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n--- HOÀN THÀNH ---")
print(f"File Master đã lưu tại: {out_path}")
print(f"Tổng số Features: {len(df_vn_master.columns)}")
display(df_vn_master.head())

Merging features for: FPT
Merging features for: VCB
Merging features for: VIC
Merging features for: VHM
Merging features for: HPG
Merging features for: MSN
Merging features for: MWG
Merging features for: TCB
Merging features for: VND
Merging features for: VNM

--- HOÀN THÀNH ---
File Master đã lưu tại: /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/vn_stocks_master_features.csv
Tổng số Features: 71


,date,open,high,low,close,volume,ticker,log_return,rsi,macd,...,quickRatio_y,currentRatio_y,pb_y,bv_y,beta_y,dividendYield_y,financialLeverage_y,ttmType_y,asset_y,marketCap_y
0,2020-03-18 07:00:00,17.72,17.75,17.48,17.50,1642800,FPT,-0.001713,33.676271,-0.660324,...,0.600221,1.400061,5.817333,19768.510267,0.910353,0.017391,1.507142,0.0,4.173432e+13,1.043684e+14
1,2020-03-19 07:00:00,17.21,17.39,17.21,17.24,1296590,FPT,-0.014969,31.776048,-0.690120,...,0.600221,1.400061,5.817333,19768.510267,0.910353,0.017391,1.507142,0.0,4.173432e+13,1.043684e+14
2,2020-03-20 07:00:00,17.42,17.44,17.12,17.24,1291360,FPT,0.000000,31.776048,-0.705599,...,0.600221,1.400061,5.817333,19768.510267,0.910353,0.017391,1.507142,0.0,4.173432e+13,1.043684e+14
3,2020-03-23 07:00:00,16.67,16.95,16.05,16.05,2826690,FPT,-0.071523,24.452171,-0.804615,...,0.600221,1.400061,5.817333,19768.510267,0.910353,0.017391,1.507142,0.0,4.173432e+13,1.043684e+14
4,2020-03-24 07:00:00,16.27,16.27,15.73,16.09,2191500,FPT,0.002489,25.077275,-0.869831,...,0.600221,1.400061,5.817333,19768.510267,0.910353,0.017391,1.507142,0.0,4.173432e+13,1.043684e+14


In [ ]:
# Kiểm tra xem có bị lặp Date + Ticker không
duplicates = df_vn_master.duplicated(subset=['date', 'ticker']).sum()
if duplicates > 0:
    print(f"❌ CẢNH BÁO: Vẫn còn {duplicates} dòng bị lặp!")
    # Nếu vẫn lặp, dùng lệnh này để xóa triệt để:
    df_vn_master = df_vn_master.drop_duplicates(subset=['date', 'ticker'])
else:
    print("Tuyệt vời: Dữ liệu không còn bị lặp ngày.")

✅ Tuyệt vời: Dữ liệu không còn bị lặp ngày.


In [5]:
import sys
from pathlib import Path
import pandas as pd

# 1. SETUP PATHS
# Find the root directory and add to sys.path for module discovery
ROOT = Path.cwd().resolve().parent 
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.preprocess import add_all_features

# Path to the directory containing Vietnam stock CSV files
DATA_DIR = ROOT / 'notebooks' / 'data' / 'vietnam' 

# 2. LOAD EXTERNAL DATASETS
sentiment_df = pd.read_csv(DATA_DIR / 'final_sentiment_features_v1.csv')
fundamentals_df = pd.read_csv(DATA_DIR / 'finance_indicators.csv')

# 3. PREPROCESS FUNDAMENTALS
# Standardize column names and strip whitespaces
fundamentals_df.columns = [c.strip() for c in fundamentals_df.columns]
if 'yearReport' in fundamentals_df.columns:
    fundamentals_df = fundamentals_df.rename(columns={'yearReport': 'year'})

# Drop non-essential columns for the model
cols_to_drop = ['sharesOutstanding', 'netProfit', 'npl']
for col in cols_to_drop:
    if col in fundamentals_df.columns:
        fundamentals_df = fundamentals_df.drop(columns=[col])

fundamentals_df['ticker'] = fundamentals_df['ticker'].str.upper()

# 4. TICKER-BY-TICKER PROCESSING
frames = []
vn_tickers = ['FPT', 'VCB', 'VIC', 'VHM', 'HPG', 'MSN', 'MWG', 'TCB', 'VND', 'VNM']

for ticker in vn_tickers:
    file_path = DATA_DIR / 'csv' / f'{ticker}_ohlcv.csv'
    if not file_path.exists():
        continue
    
    print(f"Processing features for: {ticker}")
    
    # A. Load OHLCV data
    g = pd.read_csv(file_path)
    g = g.rename(columns={'tradingDate': 'date'})
    g["date"] = pd.to_datetime(g["date"])
    g = g.sort_values("date").set_index("date")
    
    # B. Generate Technical Indicators (RSI, MACD, Log Return, etc.)
    g = add_all_features(g) 
    g = g.reset_index()
    g["ticker"] = ticker
    
    # C. Merge Sentiment Data (Key: Date, Ticker)
    t_sent = sentiment_df[sentiment_df["ticker"] == ticker].copy()
    t_sent["date"] = pd.to_datetime(t_sent["date"])
    g = pd.merge(g, t_sent, on=["date", "ticker"], how="left")
    
    # D. Merge Financial Fundamentals (Key: Year, Ticker)
    g['year'] = g['date'].dt.year
    t_fund = fundamentals_df[fundamentals_df["ticker"] == ticker].copy()

    # CRITICAL: Drop yearly duplicates to prevent data inflation (1 row per year)
    t_fund = t_fund.drop_duplicates(subset=['year'], keep='last')
    
    # Merge financial indicators (P/E, P/B, ROE, ROA, Beta, etc.)
    g = pd.merge(g, t_fund, on=["year", "ticker"], how="left")
    
    # E. Post-Merge Cleaning
    # Forward-fill financial data across daily sessions
    g = g.ffill()
    
    # Fill missing sentiment scores with 0 (for days with no news)
    sent_cols = ['sent_mean', 'sent_max', 'sent_min', 'news_count']
    for col in sent_cols:
        if col in g.columns:
            g[col] = g[col].fillna(0)
    
    frames.append(g)

# 5. CONSOLIDATION AND FINAL VALIDATION
df_vn_master = pd.concat(frames, ignore_index=True)

# Back-fill leading NaNs caused by indicator look-back windows
df_vn_master = df_vn_master.bfill()

# Remove temporary year column
if 'year' in df_vn_master.columns:
    df_vn_master = df_vn_master.drop(columns=['year'])

# --- RECTIFY DUPLICATES ---
# Ensure strict one-row-per-date-per-ticker integrity
duplicates = df_vn_master.duplicated(subset=['date', 'ticker']).sum()

if duplicates > 0:
    print(f"⚠️ Found {duplicates} duplicate entries. Dropping now...")
    df_vn_master = df_vn_master.drop_duplicates(subset=['date', 'ticker'], keep='first')
    print("✅ Duplicates removed.")
else:
    print("✅ Data integrity confirmed: No duplicates found.")

# 6. EXPORT CLEANED MASTER DATASET
out_path = DATA_DIR / "vn_stocks_master_features.csv"
df_vn_master.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n--- SUCCESS ---")
print(f"Master file saved at: {out_path}")
print(f"Total records: {len(df_vn_master):,}")
print(f"Total features: {len(df_vn_master.columns)}")
display(df_vn_master.head())

Processing features for: FPT
Processing features for: VCB
Processing features for: VIC
Processing features for: VHM
Processing features for: HPG
Processing features for: MSN
Processing features for: MWG
Processing features for: TCB
Processing features for: VND
Processing features for: VNM
✅ Data integrity confirmed: No duplicates found.

--- SUCCESS ---
Master file saved at: /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/vn_stocks_master_features.csv
Total records: 15,254
Total features: 47


,date,open,high,low,close,volume,ticker,log_return,rsi,macd,...,quickRatio,currentRatio,pb,bv,beta,dividendYield,financialLeverage,ttmType,asset,marketCap
0,2020-03-18 07:00:00,17.72,17.75,17.48,17.50,1642800,FPT,-0.001713,33.676271,-0.660324,...,1.159709,1.236096,2.587269,18803.607819,0.885645,0.04111,0.952089,1.0,3.441667e+13,3.813698e+13
1,2020-03-19 07:00:00,17.21,17.39,17.21,17.24,1296590,FPT,-0.014969,31.776048,-0.690120,...,1.159709,1.236096,2.587269,18803.607819,0.885645,0.04111,0.952089,1.0,3.441667e+13,3.813698e+13
2,2020-03-20 07:00:00,17.42,17.44,17.12,17.24,1291360,FPT,0.000000,31.776048,-0.705599,...,1.159709,1.236096,2.587269,18803.607819,0.885645,0.04111,0.952089,1.0,3.441667e+13,3.813698e+13
3,2020-03-23 07:00:00,16.67,16.95,16.05,16.05,2826690,FPT,-0.071523,24.452171,-0.804615,...,1.159709,1.236096,2.587269,18803.607819,0.885645,0.04111,0.952089,1.0,3.441667e+13,3.813698e+13
4,2020-03-24 07:00:00,16.27,16.27,15.73,16.09,2191500,FPT,0.002489,25.077275,-0.869831,...,1.159709,1.236096,2.587269,18803.607819,0.885645,0.04111,0.952089,1.0,3.441667e+13,3.813698e+13
